# 08.07 - CNN Fundamentals

**Notebook type:** Solution notebook with full working code and test cases.

**Daily output:** a CNN shape worksheet: manually compute output shapes for 5 CNN layers, then optionally verify with PyTorch.

Today is hard CV theory made practical: convolution, stride, padding, pooling, feature maps, and receptive field. You should finish able to predict CNN tensor shapes before running the model.


## Core Ideas

A convolution layer slides learnable filters over an input feature map.

Important terms:

- **kernel size:** filter window, for example `3x3`
- **stride:** how far the window moves each step
- **padding:** pixels added around the border
- **dilation:** spacing between kernel elements
- **channels:** input channels are mixed to produce output channels
- **feature map:** one output activation grid per output channel

For one spatial dimension, the output size is:

```text
out = floor((in + 2 * padding - dilation * (kernel - 1) - 1) / stride + 1)
```

For standard convolution with dilation 1:

```text
out = floor((in + 2p - k) / s + 1)
```


In [ ]:
import math

try:
    import torch
    import torch.nn as nn
    TORCH_AVAILABLE = True
except ImportError:
    torch = None
    nn = None
    TORCH_AVAILABLE = False
    print("PyTorch is not installed. Shape math tests still run; PyTorch verification is skipped.")


## Exercise 08-A: Single-Layer Output Shape

Implement output shape helpers for convolution and pooling. Use `(height, width)` tuples for spatial shapes.


In [ ]:
def _pair(x):
    if isinstance(x, tuple):
        if len(x) != 2:
            raise ValueError(f"Expected tuple of length 2, got {x}")
        return x
    return (x, x)

def conv2d_output_shape(hw, kernel_size, stride=1, padding=0, dilation=1):
    h, w = hw
    kh, kw = _pair(kernel_size)
    sh, sw = _pair(stride)
    ph, pw = _pair(padding)
    dh, dw = _pair(dilation)
    out_h = math.floor((h + 2 * ph - dh * (kh - 1) - 1) / sh + 1)
    out_w = math.floor((w + 2 * pw - dw * (kw - 1) - 1) / sw + 1)
    return out_h, out_w

def pool2d_output_shape(hw, kernel_size, stride=None, padding=0, dilation=1):
    if stride is None:
        stride = kernel_size
    return conv2d_output_shape(hw, kernel_size, stride=stride, padding=padding, dilation=dilation)

print(conv2d_output_shape((64, 64), kernel_size=3, stride=1, padding=1))
print(pool2d_output_shape((64, 64), kernel_size=2, stride=2))


## Exercise 08-B: Manual 5-Layer CNN Worksheet

Compute the shape after each layer. Use this architecture:

- input: `[3, 64, 64]`
- conv: `3 -> 16`, kernel `3`, stride `1`, padding `1`
- max pool: kernel `2`, stride `2`
- conv: `16 -> 32`, kernel `3`, stride `2`, padding `1`
- conv: `32 -> 64`, kernel `5`, stride `1`, padding `0`
- max pool: kernel `2`, stride `2`


In [ ]:
layers = [
    {"type": "conv", "name": "conv1", "out_channels": 16, "kernel_size": 3, "stride": 1, "padding": 1},
    {"type": "pool", "name": "pool1", "kernel_size": 2, "stride": 2, "padding": 0},
    {"type": "conv", "name": "conv2", "out_channels": 32, "kernel_size": 3, "stride": 2, "padding": 1},
    {"type": "conv", "name": "conv3", "out_channels": 64, "kernel_size": 5, "stride": 1, "padding": 0},
    {"type": "pool", "name": "pool2", "kernel_size": 2, "stride": 2, "padding": 0},
]

def infer_cnn_shapes(input_shape, layers):
    c, h, w = input_shape
    rows = []
    for layer in layers:
        if layer["type"] == "conv":
            h, w = conv2d_output_shape(
                (h, w),
                kernel_size=layer["kernel_size"],
                stride=layer.get("stride", 1),
                padding=layer.get("padding", 0),
                dilation=layer.get("dilation", 1),
            )
            c = layer["out_channels"]
        elif layer["type"] == "pool":
            h, w = pool2d_output_shape(
                (h, w),
                kernel_size=layer["kernel_size"],
                stride=layer.get("stride"),
                padding=layer.get("padding", 0),
                dilation=layer.get("dilation", 1),
            )
        else:
            raise ValueError(f"Unknown layer type: {layer['type']}")

        rows.append({"name": layer["name"], "type": layer["type"], "shape": (c, h, w)})
    return rows

for row in infer_cnn_shapes((3, 64, 64), layers):
    print(row)


## Exercise 08-C: Receptive Field

Receptive field tells you how much of the original image one output activation can see.

Track two numbers:

- `jump`: distance in input pixels between neighboring output activations
- `receptive_field`: size of the input region seen by one activation

Update rule for each conv/pool layer:

```text
new_receptive_field = old_receptive_field + (kernel - 1) * dilation * old_jump
new_jump = old_jump * stride
```


In [ ]:
def receptive_field_table(layers):
    receptive_field = 1
    jump = 1
    rows = []
    for layer in layers:
        kernel = _pair(layer["kernel_size"])[0]
        stride = _pair(layer.get("stride", 1))[0]
        dilation = _pair(layer.get("dilation", 1))[0]
        receptive_field = receptive_field + (kernel - 1) * dilation * jump
        jump = jump * stride
        rows.append({
            "name": layer["name"],
            "receptive_field": receptive_field,
            "jump": jump,
        })
    return rows

for row in receptive_field_table(layers):
    print(row)


## Exercise 08-D: Optional PyTorch Verification

Build the same CNN layers in PyTorch and compare the actual tensor shapes to your manual worksheet. This cell should skip gracefully if PyTorch is not installed.


In [ ]:
def build_shape_verifier_model():
    if not TORCH_AVAILABLE:
        return None
    return nn.Sequential(
        nn.Conv2d(3, 16, kernel_size=3, stride=1, padding=1),
        nn.MaxPool2d(kernel_size=2, stride=2),
        nn.Conv2d(16, 32, kernel_size=3, stride=2, padding=1),
        nn.Conv2d(32, 64, kernel_size=5, stride=1, padding=0),
        nn.MaxPool2d(kernel_size=2, stride=2),
    )

if TORCH_AVAILABLE:
    model = build_shape_verifier_model()
    x = torch.randn(1, 3, 64, 64)
    with torch.no_grad():
        y = model(x)
    print("PyTorch output shape:", tuple(y.shape))
else:
    print("Skipped PyTorch verification because torch is not installed.")


## Test Cases

Run this cell after completing the TODO cells above. A correct implementation should print `Day 08 tests passed`.


In [ ]:
def run_day08_tests():
    required_names = [
        "_pair",
        "conv2d_output_shape",
        "pool2d_output_shape",
        "infer_cnn_shapes",
        "receptive_field_table",
        "build_shape_verifier_model",
    ]
    for name in required_names:
        assert name in globals(), f"Missing function: {name}"
        assert callable(globals()[name]), f"{name} must be callable"

    assert _pair(3) == (3, 3)
    assert _pair((2, 5)) == (2, 5)
    assert conv2d_output_shape((64, 64), 3, stride=1, padding=1) == (64, 64)
    assert conv2d_output_shape((64, 64), 3, stride=2, padding=1) == (32, 32)
    assert pool2d_output_shape((64, 64), 2, stride=2) == (32, 32)

    rows = infer_cnn_shapes((3, 64, 64), layers)
    expected_shapes = [
        ("conv1", (16, 64, 64)),
        ("pool1", (16, 32, 32)),
        ("conv2", (32, 16, 16)),
        ("conv3", (64, 12, 12)),
        ("pool2", (64, 6, 6)),
    ]
    assert [(row["name"], row["shape"]) for row in rows] == expected_shapes

    rf_rows = receptive_field_table(layers)
    assert rf_rows[-1]["receptive_field"] == 28
    assert rf_rows[-1]["jump"] == 8

    if TORCH_AVAILABLE:
        model = build_shape_verifier_model()
        x = torch.randn(1, 3, 64, 64)
        with torch.no_grad():
            y = model(x)
        assert tuple(y.shape) == (1, 64, 6, 6)

    print("Day 08 tests passed")

run_day08_tests()


## Day 08 Checklist

Before coding a CNN, verify input shape, channel count, output size after each layer, final flatten size, receptive field, and whether any stride/pooling layer shrinks the image too aggressively.
